# 04 — Feature Analysis (3-Stage Band Selection)

A novel 3-stage pipeline to select the optimal spectral–temporal band subset for semantic segmentation.

| Stage | Method | Cost | Output |
|---|---|---|---|
| 1 — Filter | GSI + RF importance → joint score | Very low | Top ~25 candidate channels |
| 2 — CNN Forward Selection | Lightweight U-Net as wrapper oracle | Medium | ~10–15 selected channels |
| 3 — Full Validation | U-Net / DeepLabV3+ / SegFormer | High | mIoU comparison table |

**Assumes `02_image_processing` has been run.** Preprocessed chips (images + masks) must exist in `data/processed/`.

## Configuration

In [ ]:
import sys, os, re
from glob import glob
sys.path.append("../")

# ── Data paths (outputs of 02_image_processing.ipynb) ────────────────────────
PROCESSED_DIR    = "../data/processed"
S2_PROCESSED_DIR = os.path.join(PROCESSED_DIR, "s2")
CDL_FILTERED     = os.path.join(PROCESSED_DIR, "cdl", "cdl_2022_study_area_filtered.tif")
S2_PROCESSED     = sorted(glob(f"{S2_PROCESSED_DIR}/*_processed.tif"))

# S2 band names (GEE export order: B2, B3, B4, B5, B6, B7, B8, B11, B12)
S2_BAND_NAMES = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]

# ── MLflow tracking ───────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://mlflow-geoai.stelarea.com"
MLFLOW_EXPERIMENT   = "research-crop-mapping"

# ── Stage 1 parameters ────────────────────────────────────────────────────────
SAMPLE_FRACTION = 0.005   # 0.5% of valid pixels for GSI/RF (small-scale)
GSI_RF_ALPHA    = 0.5     # weight for GSI vs RF in joint score
TOP_K           = 25      # candidate channels passed to Stage 2

# ── Stage 2 parameters ────────────────────────────────────────────────────────
S2_ENCODER    = "resnet18"
S2_PATCH_SIZE = 128
S2_STRIDE     = 64          # patch stride (overlapping patches)
S2_MIN_VALID  = 0.3         # min fraction of non-background pixels per patch
S2_EPOCHS     = 15
S2_PATIENCE   = 5
S2_DELTA      = 0.005       # min mIoU gain to accept a new band
S2_NO_IMPROVE = 3           # stop after N consecutive non-improving bands
S2_MAX_BANDS  = 15

# ── Stage 3 parameters ────────────────────────────────────────────────────────
S3_EPOCHS     = 100
S3_PATCH_SIZE = 256

# ── Classes ───────────────────────────────────────────────────────────────────
# 11 CDL classes selected by ≥1% area threshold (see report §11)
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 61, 69, 75, 76, 220]
#   3=Rice, 6=Sunflower, 24=Winter Wheat, 36=Alfalfa, 37=Other Hay,
#   54=Tomatoes, 61=Fallow, 69=Grapes, 75=Almonds, 76=Walnuts, 220=Plums

# Remap CDL IDs → sequential 1..11 for CrossEntropyLoss (0 = background)
CLASS_REMAP = {cls_id: i + 1 for i, cls_id in enumerate(KEEP_CLASSES)}
NUM_CLASSES = len(KEEP_CLASSES) + 1   # 12 total (11 crops + background)

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"S2 processed ({len(S2_PROCESSED)} files):")
for p in S2_PROCESSED:
    print(f"  {'✅' if os.path.exists(p) else '❌'} {os.path.basename(p)}")
print(f"\nCDL filtered : {'✅' if os.path.exists(CDL_FILTERED) else '❌'} {CDL_FILTERED}")
print(f"\nMLflow URI   : {MLFLOW_TRACKING_URI}")
print(f"Experiment   : {MLFLOW_EXPERIMENT}")
print(f"NUM_CLASSES  : {NUM_CLASSES}  (classes: {KEEP_CLASSES})")


In [ ]:
import numpy as np
import rasterio
import pandas as pd

S2_NODATA = -9999.0

# ── Stack all processed S2 files into a single multi-temporal array ───────────
all_arrays    = []
all_bandnames = []

for s2_path in S2_PROCESSED:
    fname = os.path.basename(s2_path)
    # Parse date from "S2H_2022_2022_01_01_processed.tif" → "20220101"
    m = re.search(r"_(\d{4})_(\d{2})_(\d{2})_processed", fname)
    date_str = f"{m.group(1)}{m.group(2)}{m.group(3)}" if m else fname[:8]

    with rasterio.open(s2_path) as src:
        arr = src.read().astype(np.float32)   # (9, H, W)
    arr[arr == S2_NODATA] = np.nan
    all_arrays.append(arr)
    all_bandnames.extend([f"{b}_{date_str}" for b in S2_BAND_NAMES])

stacked    = np.concatenate(all_arrays, axis=0)   # (n_channels, H, W)
n_channels, H, W = stacked.shape
band_names = all_bandnames   # alias used by band_selection.py helpers

print(f"Stacked S2 : {n_channels} channels × {H} × {W} px")
print(f"Band names : {all_bandnames}")

# ── Load CDL filtered label into memory ───────────────────────────────────────
with rasterio.open(CDL_FILTERED) as src:
    cdl = src.read(1).astype(np.int32)   # (H, W)

assert cdl.shape == (H, W), f"Shape mismatch: CDL {cdl.shape} vs S2 ({H},{W})"
print(f"\nCDL shape  : {cdl.shape}")

unique, counts = np.unique(cdl[cdl != 0], return_counts=True)
print(f"Active CDL classes: {dict(zip(unique.tolist(), counts.tolist()))}")

# ── Flatten + filter to labeled pixels ────────────────────────────────────────
img_2d = stacked.reshape(n_channels, -1).T   # (N_pixels, n_channels)
lbl_1d = cdl.flatten()                        # (N_pixels,)

valid_mask = np.isin(lbl_1d, KEEP_CLASSES) & np.isfinite(img_2d).all(axis=1)
img_valid  = img_2d[valid_mask]
lbl_valid  = lbl_1d[valid_mask]

print(f"\nValid labeled pixels : {len(lbl_valid):,}  "
      f"({100*len(lbl_valid)/len(lbl_1d):.1f}% of {len(lbl_1d):,} total)")

# ── Sample fraction for Stage 1 ───────────────────────────────────────────────
rng = np.random.default_rng(42)
n   = max(1000, int(len(lbl_valid) * SAMPLE_FRACTION))
idx = rng.choice(len(lbl_valid), n, replace=False)

# Stage 1 uses raw CDL IDs (GSI/RF work on any integer class labels)
df        = pd.DataFrame(img_valid[idx], columns=all_bandnames)
df.insert(0, "class_label", lbl_valid[idx].astype(int))
CLASS_COL = "class_label"

print(f"Sampled    : {len(df):,} pixels  (SAMPLE_FRACTION={SAMPLE_FRACTION})")
print(f"\nClass distribution:")
print(df[CLASS_COL].value_counts().sort_index().to_string())

# Free the full-resolution stacked array (not needed anymore for Stage 1)
del stacked, img_2d


---
# Stage 1 — Filter Preselection
**Goal:** Reduce all channels to ~25 candidates using cheap, non-spatial statistics.

$$\text{Score}_b = \alpha \cdot \text{GSI}_{\text{norm},b} + (1-\alpha) \cdot \text{RF}_{\text{norm},b}$$

## 1.1 — Start Stage 1 MLflow Run

In [ ]:
import mlflow
from datetime import datetime

# Validate data is ready
assert df is not None and len(df) > 0, "Run the stacking cell first."
print(f"Stage 1 input  : {len(df):,} pixels × {len(all_bandnames)} channels")
print(f"Classes present: {sorted(df[CLASS_COL].unique())}")

# ── Connect to remote MLflow and start Stage 1 run ───────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
stage1_run = mlflow.start_run(run_name=f"stage1_gsi_rf_{RUN_TIMESTAMP}")

mlflow.log_params({
    "stage":             "1_filter_preselection",
    "n_images":          len(S2_PROCESSED),
    "n_bands_per_image": len(S2_BAND_NAMES),
    "total_channels":    n_channels,
    "sample_fraction":   SAMPLE_FRACTION,
    "n_sampled_pixels":  len(df),
    "gsi_rf_alpha":      GSI_RF_ALPHA,
    "top_k":             TOP_K,
    "keep_classes":      str(KEEP_CLASSES),
    "num_classes":       NUM_CLASSES,
})

print(f"\nMLflow tracking URI : {MLFLOW_TRACKING_URI}")
print(f"Experiment          : {MLFLOW_EXPERIMENT}")
print(f"Stage 1 run ID      : {stage1_run.info.run_id}")


## 1.2 — Compute GSI

In [ ]:
print("Computing GSI...")
gsi_df   = bs.calculate_gsi(df, CLASS_COL)
gsi_mean = gsi_df.mean(axis=1).sort_values(ascending=False)

print("\nGSI ranking (top 10):")
print(gsi_mean.head(10).to_string())
bs.plot_gsi(gsi_df, save_path="gsi_plot.png")

## 1.3 — Compute RF Feature Importance

In [ ]:
import matplotlib.pyplot as plt

print("Computing RF feature importance...")
rf_importance = bs.compute_rf_importance(df, CLASS_COL, n_estimators=200)

print("\nRF importance ranking (top 10):")
print(rf_importance.head(10).to_string())

rf_importance.sort_values().plot(kind="barh", figsize=(8, 5), color="steelblue")
plt.title("RF Feature Importance per Band")
plt.xlabel("Normalized Importance")
plt.tight_layout()
plt.show()

## 1.4 — Compute Joint Score & Select Top K

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

joint_score  = bs.compute_joint_score(gsi_mean, rf_importance, alpha=GSI_RF_ALPHA)
stage1_bands = bs.select_top_k(joint_score, k=TOP_K)

# Summary table
gsi_norm = (gsi_mean - gsi_mean.min()) / (gsi_mean.max() - gsi_mean.min() + 1e-9)
summary  = pd.DataFrame({
    "GSI (norm)":  gsi_norm,
    "RF (norm)":   rf_importance,
    "Joint Score": joint_score,
}).sort_values("Joint Score", ascending=False)
summary["Selected"] = summary.index.isin(stage1_bands)

print(f"Stage 1 selected {len(stage1_bands)} bands (top-{TOP_K}):\n  {stage1_bands}")
print()
print(summary.to_string())

# Plot joint score ranking
fig, ax = plt.subplots(figsize=(8, 5))
summary["Joint Score"].sort_values().plot(kind="barh", color="darkorange", ax=ax)
ax.axvline(joint_score[stage1_bands[-1]], color="red", linestyle="--",
           label=f"Top-{TOP_K} cutoff")
ax.set_title(f"Joint Score per Band (α={GSI_RF_ALPHA})")
ax.set_xlabel("Joint Score")
ax.legend()
plt.tight_layout()
plt.savefig("stage1_joint_score.png", dpi=150)
plt.show()

# ── Log to MLflow ─────────────────────────────────────────────────────────────
for band in all_bandnames:
    mlflow.log_metric(f"gsi_{band}",   float(gsi_norm.get(band, 0)))
    mlflow.log_metric(f"rf_{band}",    float(rf_importance.get(band, 0)))
    mlflow.log_metric(f"joint_{band}", float(joint_score.get(band, 0)))

mlflow.set_tag("stage1_selected_bands", str(stage1_bands))
mlflow.set_tag("stage1_n_selected",     str(len(stage1_bands)))
mlflow.set_tag("stage1_top_k",          str(TOP_K))

summary_csv = "stage1_band_scores.csv"
summary.to_csv(summary_csv)
mlflow.log_artifact(summary_csv)
mlflow.log_artifact("gsi_plot.png")
mlflow.log_artifact("stage1_joint_score.png")

mlflow.end_run()
print(f"\n✅ Stage 1 logged to MLflow  (run_id: {stage1_run.info.run_id})")
print(f"   Selected bands: {stage1_bands}")


---
# Stage 2 — CNN Forward Selection
**Goal:** Use a lightweight U-Net as the selection oracle. Iteratively add bands, keeping only those that raise validation mIoU by at least δ.

$$\text{keep band } b \iff \text{mIoU}_{\text{new}} > \text{mIoU}_{\text{prev}} + \delta$$

> ⚠️ Each iteration trains a small model for ~15 epochs. Expect ~6–10 hours total.

## 2.1 — Setup

In [ ]:
import time
import numpy as np
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader, random_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Map band name → index in the stacked n_channels array
band_index = {name: i for i, name in enumerate(all_bandnames)}
print(f"Band index map ({len(band_index)} entries): first 5 = {list(band_index.items())[:5]}")

# Lookup table for CDL ID → model sequential ID (0=background, 1-11=crops)
# Using LUT indexed by CDL ID for fast vectorized remapping
REMAP_LUT = np.zeros(256, dtype=np.int64)   # all → 0 (background)
for cdl_id, model_id in CLASS_REMAP.items():
    if cdl_id < 256:
        REMAP_LUT[cdl_id] = model_id

print(f"\nClass remap (CDL→model): {CLASS_REMAP}")
print(f"NUM_CLASSES: {NUM_CLASSES}")


In [ ]:
from torch.utils.data import Dataset

class RasterPatchDataset(Dataset):
    """
    Generate image/mask patch pairs on-the-fly from processed rasters.

    Avoids pre-generating chip files; works directly with the S2 and CDL
    outputs of 02_image_processing.ipynb.

    - CDL is loaded into memory once to build the valid patch index cheaply.
    - S2 files are kept open and accessed via rasterio windows per-patch.
    - CDL class IDs are remapped to sequential model IDs via REMAP_LUT.
    """

    def __init__(self, s2_paths, cdl_path, patch_size, stride,
                 min_valid_frac=0.3, band_indices=None):
        self.s2_paths     = s2_paths
        self.cdl_path     = cdl_path
        self.patch_size   = patch_size
        self.band_indices = band_indices

        # Load CDL into memory to build the patch index (cheap: ~26 MB uint8)
        with rasterio.open(cdl_path) as src:
            self._cdl   = src.read(1).astype(np.int32)
            self.height = src.height
            self.width  = src.width

        # Open S2 rasterio sources once and reuse across all __getitem__ calls
        # (num_workers=0 required — rasterio handles cannot be pickled)
        self._s2_srcs = [rasterio.open(p) for p in s2_paths]

        # Build list of valid patch top-left corners (row, col)
        ps = patch_size
        self.patches = [
            (r, c)
            for r in range(0, self.height - ps + 1, stride)
            for c in range(0, self.width  - ps + 1, stride)
            if np.isin(self._cdl[r:r+ps, c:c+ps], KEEP_CLASSES).mean() >= min_valid_frac
        ]
        print(f"RasterPatchDataset: {len(self.patches)} valid patches  "
              f"(patch={ps}px, stride={stride}px, min_valid={min_valid_frac})")

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        r, c = self.patches[idx]
        ps   = self.patch_size
        win  = rasterio.windows.Window(c, r, ps, ps)

        # Read all temporal S2 files and stack along the channel axis
        arrays = [src.read(window=win).astype(np.float32) for src in self._s2_srcs]
        img    = np.concatenate(arrays, axis=0)   # (n_channels, ps, ps)

        # Select band subset if specified
        if self.band_indices is not None:
            img = img[self.band_indices]

        # Replace nodata sentinel with 0 before normalization
        img[img == S2_NODATA] = 0.0

        # Per-channel min-max normalization
        for ch in range(img.shape[0]):
            mn, mx = img[ch].min(), img[ch].max()
            img[ch] = (img[ch] - mn) / (mx - mn + 1e-9)

        # Remap CDL IDs → sequential model class IDs (0-11)
        cdl_patch = self._cdl[r:r+ps, c:c+ps]
        mask      = REMAP_LUT[np.clip(cdl_patch, 0, 255)]

        return torch.from_numpy(img), torch.from_numpy(mask.astype(np.int64))

    def __del__(self):
        for src in getattr(self, "_s2_srcs", []):
            try:
                src.close()
            except Exception:
                pass


In [ ]:
def build_unet(in_channels, num_classes, encoder=S2_ENCODER):
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights=None,   # no pretrained weights — channel count varies
        in_channels=in_channels,
        classes=num_classes,
    ).to(DEVICE)


def compute_miou(preds, labels, num_classes, ignore_index=0):
    """Mean IoU over foreground classes (class 0 = background, skipped)."""
    preds_np  = preds.view(-1).cpu().numpy()
    labels_np = labels.view(-1).cpu().numpy()
    ious = []
    for cls in range(1, num_classes):
        p = preds_np  == cls
        l = labels_np == cls
        intersection = (p & l).sum()
        union        = (p | l).sum()
        if union > 0:
            ious.append(intersection / union)
    return float(np.mean(ious)) if ious else 0.0


def train_eval(band_indices, num_classes, epochs=S2_EPOCHS, patience=S2_PATIENCE):
    """
    Train a lightweight U-Net on the given band subset.
    Uses RasterPatchDataset to generate patches on-the-fly from processed rasters.
    Returns best validation mIoU.
    """
    dataset = RasterPatchDataset(
        s2_paths=S2_PROCESSED,
        cdl_path=CDL_FILTERED,
        patch_size=S2_PATCH_SIZE,
        stride=S2_STRIDE,
        min_valid_frac=S2_MIN_VALID,
        band_indices=band_indices,
    )

    if len(dataset) < 4:
        print(f"  ⚠️  Only {len(dataset)} patches — skipping.")
        return 0.0

    n_val   = max(1, int(0.2 * len(dataset)))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42),
    )

    # num_workers=0: rasterio file handles cannot be pickled for forked workers
    train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)

    model     = build_unet(len(band_indices), num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

    best_miou, no_improve = 0.0, 0

    for epoch in range(epochs):
        model.train()
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(imgs), masks).backward()
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, masks in val_dl:
                preds = model(imgs.to(DEVICE)).argmax(dim=1)
                all_preds.append(preds.cpu())
                all_labels.append(masks)

        miou = compute_miou(
            torch.cat(all_preds), torch.cat(all_labels), num_classes
        )

        if miou > best_miou + 1e-4:
            best_miou, no_improve = miou, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    return best_miou


## 2.2 — Forward Selection Loop

In [ ]:
candidates = stage1_bands   # ordered by joint score (descending from Stage 1)
selected   = []
prev_miou  = 0.0
no_improve = 0
history    = []

print(f"Forward selection — {len(candidates)} candidates  "
      f"(δ={S2_DELTA}, patience={S2_NO_IMPROVE}, max_bands={S2_MAX_BANDS})\n")

# ── Start Stage 2 MLflow run ─────────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

stage2_run = mlflow.start_run(run_name=f"stage2_cnn_fwd_{RUN_TIMESTAMP}")
mlflow.log_params({
    "stage":         "2_cnn_forward_selection",
    "encoder":       S2_ENCODER,
    "patch_size":    S2_PATCH_SIZE,
    "stride":        S2_STRIDE,
    "min_valid":     S2_MIN_VALID,
    "epochs":        S2_EPOCHS,
    "patience":      S2_PATIENCE,
    "delta":         S2_DELTA,
    "no_improve":    S2_NO_IMPROVE,
    "max_bands":     S2_MAX_BANDS,
    "candidates":    str(candidates),
    "num_classes":   NUM_CLASSES,
    "stage1_run_id": stage1_run.info.run_id,
})

# ── Forward selection loop ────────────────────────────────────────────────────
for step, band in enumerate(candidates):
    if len(selected) >= S2_MAX_BANDS:
        print(f"Reached max_bands={S2_MAX_BANDS}. Stopping.")
        break
    if no_improve >= S2_NO_IMPROVE:
        print(f"No improvement for {S2_NO_IMPROVE} consecutive bands. Stopping.")
        break

    trial_bands   = selected + [band]
    trial_indices = [band_index[b] for b in trial_bands]

    t0      = time.time()
    miou    = train_eval(trial_indices, NUM_CLASSES)
    elapsed = time.time() - t0

    gain     = miou - prev_miou
    accepted = gain >= S2_DELTA

    if accepted:
        selected   = trial_bands
        prev_miou  = miou
        no_improve = 0
        tag = "✅ accepted"
    else:
        no_improve += 1
        tag = "❌ rejected"

    history.append({
        "step": step, "band": band, "n_bands": len(trial_bands),
        "mIoU": round(miou, 4), "gain": round(gain, 4),
        "accepted": accepted, "elapsed_s": round(elapsed),
    })

    # Log per-step metrics (step = iteration number)
    mlflow.log_metrics({"miou": miou, "gain": gain, "accepted": int(accepted)},
                        step=step)

    print(f"  {tag}  +{band:<22}  mIoU={miou:.4f}  gain={gain:+.4f}  ({elapsed:.0f}s)")

stage2_bands = selected

# ── Log final results ─────────────────────────────────────────────────────────
mlflow.set_tag("stage2_selected_bands", str(stage2_bands))
mlflow.set_tag("stage2_n_selected",     str(len(stage2_bands)))
mlflow.log_metric("final_miou",  prev_miou)
mlflow.log_metric("n_selected",  len(stage2_bands))

hist_df  = pd.DataFrame(history)
hist_csv = "stage2_forward_selection_history.csv"
hist_df.to_csv(hist_csv, index=False)
mlflow.log_artifact(hist_csv)

mlflow.end_run()

print(f"\n✅ Stage 2 complete — {len(stage2_bands)} bands selected: {stage2_bands}")
print(f"   Final mIoU    : {prev_miou:.4f}")
print(f"   MLflow run ID : {stage2_run.info.run_id}")
print(f"\n{hist_df.to_string(index=False)}")


## 2.3 — Plot mIoU vs Band Count

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history)

accepted_df = hist_df[hist_df["accepted"]]
rejected_df = hist_df[~hist_df["accepted"]]

plt.figure(figsize=(10, 5))
plt.plot(accepted_df["n_bands"], accepted_df["mIoU"], "o-", color="green",  label="Accepted band")
plt.scatter(rejected_df["n_bands"], rejected_df["mIoU"], marker="x", color="red", s=80, label="Rejected band")
plt.axhline(prev_miou, linestyle="--", color="gray", label=f"Final mIoU={prev_miou:.4f}")
plt.xlabel("Number of bands")
plt.ylabel("Validation mIoU")
plt.title("CNN Forward Selection: mIoU vs Band Count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("forward_selection_curve.png", dpi=150)
plt.show()

print(hist_df.to_string(index=False))

# Save selection history
stage2_bands = selected
hist_df.to_csv("forward_selection_history.csv", index=False)
print("\nSaved: forward_selection_history.csv")

---
# Stage 3 — Full Model Validation
**Goal:** Train full segmentation models on three band sets and compare performance.

| Experiment | Band Set | Description |
|---|---|---|
| A — Baseline | All bands | No selection |
| B — Stage 1 | Top-K filter bands | GSI + RF only |
| C — Stage 2 | CNN-selected bands | Forward selection |

Each trained with: U-Net · DeepLabV3+ · SegFormer

## 3.1 — Define Experiments

In [ ]:
all_band_indices    = list(range(len(band_names)))           # Exp A
stage1_band_indices = [band_index[b] for b in stage1_bands]  # Exp B
stage2_band_indices = [band_index[b] for b in stage2_bands]  # Exp C

experiments = {
    "A_all":     {"bands": all_band_indices,    "label": f"All ({len(all_band_indices)} bands)"},
    "B_stage1":  {"bands": stage1_band_indices, "label": f"Stage1 Top-{TOP_K} ({len(stage1_band_indices)} bands)"},
    "C_stage2":  {"bands": stage2_band_indices, "label": f"Stage2 CNN-selected ({len(stage2_band_indices)} bands)"},
}

models = ["unet", "deeplabv3plus", "segformer"]

print("Experiments:")
for key, exp in experiments.items():
    print(f"  {key}: {exp['label']}")

## 3.2 — Train & Evaluate

> ⚠️ This will take ~27 GPU hours (3 models × 3 experiments × ~3 hours each). Run on a GPU server.

In [ ]:
import geoai

results = []

for exp_key, exp in experiments.items():
    for model_name in models:
        band_indices = exp["bands"]
        n_channels   = len(band_indices)
        print(f"\n─── {exp_key} | {model_name} | {n_channels} channels ───")

        out_dir = f"../models/{exp_key}_{model_name}"
        os.makedirs(out_dir, exist_ok=True)

        t_start  = time.time()
        mem_before = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

        geoai.train_segmentation_model(
            images_dir=IMAGES_DIR,
            labels_dir=MASKS_DIR,
            output_dir=out_dir,
            architecture=model_name,
            num_channels=n_channels,
            num_classes=NUM_CLASSES,
            batch_size=8,
            num_epochs=S3_EPOCHS,
            learning_rate=1e-4,
            val_split=0.2,
        )

        elapsed   = time.time() - t_start
        mem_after = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

        # Load best val mIoU from training log
        log_path = os.path.join(out_dir, "training_log.json")
        if os.path.exists(log_path):
            with open(log_path) as f:
                log = json.load(f)
            best_miou = max(entry.get("val_miou", 0) for entry in log)
        else:
            best_miou = None

        results.append({
            "experiment":   exp_key,
            "band_label":   exp["label"],
            "model":        model_name,
            "n_channels":   n_channels,
            "val_mIoU":     best_miou,
            "train_time_s": round(elapsed),
            "gpu_mem_mb":   round((mem_after - mem_before) / 1e6, 1),
        })

        print(f"  mIoU={best_miou:.4f}  time={elapsed/3600:.1f}h  mem={results[-1]['gpu_mem_mb']}MB")

## 3.3 — Compare Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("stage3_results.csv", index=False)

# Pivot table: models as columns, experiments as rows
pivot = results_df.pivot_table(
    index=["experiment", "band_label"],
    columns="model",
    values="val_mIoU"
).round(4)

print("\nmIoU Comparison Table")
print(pivot.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mIoU comparison
for model_name, grp in results_df.groupby("model"):
    axes[0].plot(grp["experiment"], grp["val_mIoU"], marker="o", label=model_name)
axes[0].set_title("Validation mIoU by Experiment & Model")
axes[0].set_ylabel("mIoU")
axes[0].set_xlabel("Experiment")
axes[0].legend()
axes[0].grid(True)

# Training time comparison
for model_name, grp in results_df.groupby("model"):
    axes[1].plot(grp["experiment"], grp["train_time_s"] / 3600, marker="s", label=model_name)
axes[1].set_title("Training Time (hours) by Experiment & Model")
axes[1].set_ylabel("Hours")
axes[1].set_xlabel("Experiment")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("stage3_comparison.png", dpi=150)
plt.show()